# Section 6: Jev-Like APIs Across Providers

**What you'll learn**
- Send the same System One request to decision models from several providers
- Switch providers with `model=` and servers with `base_url=`
- Run the same request on local servers (Ollaya and Ollama)
- Benchmark every model on 200 real card messages: accuracy, confidence, tokens, time, cost
- Find the messages where models disagree, and check whether agreement predicts correctness
- Call OpenRouter's native decisions endpoint with raw HTTP

## Setup
- Slides 1-4
- `.env` points the SDK at OpenRouter, so one key reaches every hosted model below
- `card_messages.csv`: 200 real bank customer messages, 10 card intents, 20 each
- `timeout=60`: some hosted providers answer slower than the SDK's 10 second default

In [1]:
import os, time, asyncio
import httpx
import pandas as pd
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, AsyncTypeSafeClient, Choice, RetryPolicy

load_dotenv()
client = TypeSafeClient(timeout=60)

cards = pd.read_csv("data/card_messages.csv")
cards.head()

,id,text,intent
0,1,I'm starting to think my card is lost because ...,card_arrival
1,2,What do I need to do to change my PIN?,change_pin
2,3,What do I do if I think my card was improperly...,compromised_card
3,4,I cannot find my credit card.,lost_or_stolen_card
4,5,How can I change my Tholepin ?,change_pin


## One Routing Question
- The 10 intents become the options of one `Choice`
- The same question object goes to every provider

In [2]:
INTENT = Choice(
    instructions="What does the customer need help with?",
    criteria={
        "card_arrival": "the card has not arrived yet, or asks when it will arrive",
        "activate_my_card": "how to activate a card they already have",
        "change_pin": "change, reset or unblock the PIN",
        "card_about_to_expire": "the card is expiring soon, or asks about a replacement "
                                "for expiry",
        "lost_or_stolen_card": "the card is lost, missing or stolen",
        "compromised_card": "someone else may have used the card or its details",
        "card_not_working": "the card does not work at all, damaged or not accepted "
                            "anywhere",
        "declined_card_payment": "one card payment was declined or rejected",
        "card_payment_fee_charged": "an unexpected fee on a card payment",
        "card_payment_wrong_exchange_rate": "a card payment abroad used a wrong or bad "
                                            "exchange rate",
    },
)

## Same Request, Different Provider
- Slides 5-7
- `model=` is the only change; `r.model` is the exact dated version that answered

In [3]:
msg = cards["text"][0]
msg

"I'm starting to think my card is lost because it still hasn't arrived, can you help?"

In [4]:
r = client.system_one(msg, {"intent": INTENT}, model="liquid/d1")
r

SystemOneResponse(model='liquid/d1-20260930', usage=Usage(input_tokens=174, output_tokens=0), answers={'intent': ChoiceAnswer(type='choice', choice='card_arrival', confidence=0.7736000553509075, probabilities={'card_arrival': 0.7962400498158168, 'activate_my_card': 0.00041370615847260127, 'change_pin': 0.00036509440343224236, 'card_about_to_expire': 0.00046879049349744746, 'lost_or_stolen_card': 0.2013210063591487, 'compromised_card': 0.00023572277598444944, 'card_not_working': 0.00046879049349744746, 'declined_card_payment': 0.00023572277598444944, 'card_payment_fee_charged': 0.0001835810825244546, 'card_payment_wrong_exchange_rate': 6.75356416417072e-05})})

In [5]:
HOSTED = [
    "jev-latest",
    "liquid/d1",
    "upstage/solar-decide",
    "jaredpalmer/kev-4b",
    "perplexity/pplx-decider-v1.2",
    "cloudflare/clef",
]

answers = [client.system_one(msg, {"intent": INTENT}, model=m) for m in HOSTED]
pd.DataFrame({
    "asked": HOSTED,
    "answered_by": [a.model for a in answers],
    "choice": [a.answers["intent"].choice for a in answers],
    "confidence": [a.answers["intent"].confidence for a in answers],
})

,asked,answered_by,choice,confidence
0,jev-latest,typesafe/jev-1.14-20261014,card_arrival,0.900000
1,liquid/d1,liquid/d1-20260930,card_arrival,0.725628
2,upstage/solar-decide,upstage/solar-decide-20260928,card_arrival,0.997999
3,jaredpalmer/kev-4b,jaredpalmer/kev-4b-20260924,card_arrival,0.861300
4,perplexity/pplx-decider-v1.2,perplexity/pplx-decider-v1.2-20261016,card_arrival,0.909922
5,cloudflare/clef,cloudflare/clef,card_arrival,0.899900


In [6]:
answers[0].answers["intent"].probabilities

{'card_arrival': 0.91,
 'activate_my_card': 0.0,
 'change_pin': 0.0,
 'card_about_to_expire': 0.0,
 'lost_or_stolen_card': 0.09,
 'compromised_card': 0.0,
 'card_not_working': 0.0,
 'declined_card_payment': 0.0,
 'card_payment_fee_charged': 0.0,
 'card_payment_wrong_exchange_rate': 0.0}

## Same API on Local Servers
- Slides 8-9
- `base_url` switches the server; the path `/v1/systemone` stays the same
- Local servers ignore the key, but the SDK needs a non-empty one

In [7]:
ollaya = TypeSafeClient(base_url="http://localhost:11435", api_key="local")
ollaya.system_one(msg, {"intent": INTENT}, model="laya")

SystemOneResponse(model='laya:en', usage=Usage(input_tokens=193, output_tokens=0), answers={'intent': ChoiceAnswer(type='choice', choice='lost_or_stolen_card', confidence=0.9916, probabilities={'card_arrival': 0.0055, 'activate_my_card': 0.0, 'change_pin': 0.0, 'card_about_to_expire': 0.0, 'lost_or_stolen_card': 0.9924, 'compromised_card': 0.0007, 'card_not_working': 0.0002, 'declined_card_payment': 0.0009, 'card_payment_fee_charged': 0.0001, 'card_payment_wrong_exchange_rate': 0.0002})})

In [8]:
ollama = TypeSafeClient(base_url="http://localhost:11434", api_key="local")
ollama.system_one(msg, {"intent": INTENT}, model="nimble")

SystemOneResponse(model='nimble', usage=Usage(input_tokens=443, output_tokens=0), answers={'intent': ChoiceAnswer(type='choice', choice='card_arrival', confidence=0.9531522333545668, probabilities={'card_arrival': 0.9578370100191101, 'activate_my_card': 0.0006130264109246568, 'change_pin': 0.00025308571489530697, 'card_about_to_expire': 0.0004705860944523572, 'lost_or_stolen_card': 0.040363725109505255, 'compromised_card': 0.00011403475004811337, 'card_not_working': 0.0001587246155049107, 'declined_card_payment': 3.065032565708611e-05, 'card_payment_fee_charged': 4.507489291462664e-05, 'card_payment_wrong_exchange_rate': 0.00011408206698746128})})

## Benchmark: 200 Messages per Model
- Slide 10
- `Semaphore(8)`: at most 8 requests in flight, providers rate-limit bigger bursts with `429`

In [9]:
gate = asyncio.Semaphore(8)

async def decide(aclient, model, text):
    async with gate:
        return await aclient.system_one(text, {"intent": INTENT}, model=model)

- One client per server: the hosted models share one, each local server gets its own
- `RetryPolicy(max_retries=6)`: on a `429` the SDK waits as long as the provider asks, then tries again

In [10]:
hosted = AsyncTypeSafeClient(timeout=60, retry=RetryPolicy(max_retries=6))
CLIENTS = {m: hosted for m in HOSTED}
CLIENTS["laya"] = AsyncTypeSafeClient(base_url="http://localhost:11435", api_key="local")
CLIENTS["nimble"] = AsyncTypeSafeClient(
    base_url="http://localhost:11434",
    api_key="local",
)

- One model at a time; `results` keeps every response, `seconds` the wall time per model

In [11]:
results, seconds = {}, {}

for m, aclient in CLIENTS.items():
    start = time.perf_counter()
    results[m] = await asyncio.gather(
        *[decide(aclient, m, text) for text in cards["text"]]
    )
    seconds[m] = time.perf_counter() - start

- One column per model, one row per message

In [12]:
preds = pd.DataFrame(
    {m: [r.answers["intent"].choice for r in rs] for m, rs in results.items()}
)
conf = pd.DataFrame(
    {m: [r.answers["intent"].confidence for r in rs] for m, rs in results.items()}
)
tokens = pd.DataFrame(
    {m: [r.usage.input_tokens for r in rs] for m, rs in results.items()}
)
preds.head()

,jev-latest,liquid/d1,upstage/solar-decide,jaredpalmer/kev-4b,perplexity/pplx-decider-v1.2,cloudflare/clef,laya,nimble
0,card_arrival,card_arrival,card_arrival,card_arrival,card_arrival,card_arrival,lost_or_stolen_card,card_arrival
1,change_pin,change_pin,change_pin,change_pin,change_pin,change_pin,change_pin,change_pin
2,compromised_card,compromised_card,compromised_card,compromised_card,compromised_card,compromised_card,compromised_card,compromised_card
3,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card
4,change_pin,change_pin,change_pin,change_pin,change_pin,change_pin,change_pin,change_pin


## The Benchmark Table
- Slide 11
- Each provider counts input tokens its own way, so compare tokens per message, not per token

In [13]:
bench = pd.DataFrame({
    "accuracy": preds.eq(cards["intent"], axis=0).mean(),
    "mean_confidence": conf.mean(),
    "tokens_per_message": tokens.mean(),
    "seconds_for_200": pd.Series(seconds),
})
bench

,accuracy,mean_confidence,tokens_per_message,seconds_for_200
jev-latest,0.970,0.968800,537.590,19.029869
liquid/d1,0.970,0.963567,166.450,22.538629
upstage/solar-decide,0.945,0.946341,517.165,43.111673
jaredpalmer/kev-4b,0.980,0.871755,190.575,27.192702
perplexity/pplx-decider-v1.2,0.975,0.953663,258.575,21.364793
cloudflare/clef,0.975,0.947354,385.575,22.421374
laya,0.865,0.833188,185.590,2.130186
nimble,0.960,0.967627,435.735,8.725362


## Cost per Million Messages
- Slides 12-13
- Input price in dollars per million tokens, output is free on all of them
- Local models have no per-token price: their cost is your GPU, so they show `NaN`

In [14]:
# OpenRouter's published input prices, from
# GET https://openrouter.ai/api/v1/models/<id>/endpoints
# (pricing.prompt x 1,000,000, the cheapest endpoint,
# which OpenRouter routes to by default)
PRICE = {
    "jev-latest": 0.042,
    "liquid/d1": 0.02,
    "upstage/solar-decide": 0.05,
    "jaredpalmer/kev-4b": 0.042,
    "perplexity/pplx-decider-v1.2": 0.02,
    "cloudflare/clef": 0.042,
}

bench["usd_per_million_messages"] = bench["tokens_per_message"] * bench.index.map(PRICE)
bench

,accuracy,mean_confidence,tokens_per_message,seconds_for_200,usd_per_million_messages
jev-latest,0.970,0.968800,537.590,19.029869,22.57878
liquid/d1,0.970,0.963567,166.450,22.538629,3.32900
upstage/solar-decide,0.945,0.946341,517.165,43.111673,25.85825
jaredpalmer/kev-4b,0.980,0.871755,190.575,27.192702,8.00415
perplexity/pplx-decider-v1.2,0.975,0.953663,258.575,21.364793,5.17150
cloudflare/clef,0.975,0.947354,385.575,22.421374,16.19415
laya,0.865,0.833188,185.590,2.130186,NaN
nimble,0.960,0.967627,435.735,8.725362,NaN


## Where Models Disagree
- Slide 14
- `nunique(axis=1)`: how many different answers each message got across the 8 models

In [15]:
n_answers = preds.nunique(axis=1)
n_answers.value_counts()

1    168
2     26
3      6
Name: count, dtype: int64

In [16]:
disagree = pd.concat([cards[["text", "intent"]], preds], axis=1)[n_answers > 1]
disagree.head(10)

,text,intent,jev-latest,liquid/d1,upstage/solar-decide,jaredpalmer/kev-4b,perplexity/pplx-decider-v1.2,cloudflare/clef,laya,nimble
0,I'm starting to think my card is lost because ...,card_arrival,card_arrival,card_arrival,card_arrival,card_arrival,card_arrival,card_arrival,lost_or_stolen_card,card_arrival
14,I think I lost my card . I dont know how long ...,lost_or_stolen_card,lost_or_stolen_card,compromised_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,compromised_card
29,How can I get my physical card to work?,card_not_working,card_not_working,card_not_working,activate_my_card,activate_my_card,card_not_working,card_not_working,card_not_working,activate_my_card
34,When will my card be activated?,activate_my_card,activate_my_card,activate_my_card,card_arrival,activate_my_card,activate_my_card,activate_my_card,card_arrival,activate_my_card
50,My card didn't work in one of the shops.,declined_card_payment,declined_card_payment,declined_card_payment,card_not_working,declined_card_payment,declined_card_payment,declined_card_payment,card_not_working,declined_card_payment
58,I broke my card,card_not_working,card_not_working,card_not_working,card_not_working,card_not_working,card_not_working,card_not_working,card_arrival,card_not_working
62,going to need a new card what are the fees and...,card_about_to_expire,card_about_to_expire,card_about_to_expire,card_arrival,card_about_to_expire,card_about_to_expire,card_about_to_expire,card_payment_fee_charged,card_about_to_expire
65,Are you able to locate my card?,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card,card_arrival
68,My card doesn't accept any transaction at all....,card_not_working,card_not_working,card_not_working,card_not_working,card_not_working,card_not_working,card_not_working,declined_card_payment,card_not_working
84,"Someone has my card number, freeze my account.",compromised_card,compromised_card,compromised_card,compromised_card,compromised_card,compromised_card,compromised_card,lost_or_stolen_card,compromised_card


In [17]:
disagree["intent"].value_counts()

intent
card_not_working                    8
compromised_card                    7
card_about_to_expire                4
card_arrival                        3
lost_or_stolen_card                 3
activate_my_card                    3
declined_card_payment               2
card_payment_fee_charged            1
card_payment_wrong_exchange_rate    1
Name: count, dtype: int64

## Does Agreement Predict Correctness?
- Slide 15
- `mode(axis=1)` gives the majority answer per message
- Compare messages where every model agreed with the rest

In [18]:
majority = preds.mode(axis=1)[0]
pd.crosstab(
    n_answers == 1,
    majority == cards["intent"],
    rownames=["all_agree"],
    colnames=["majority_right"],
)

majority_right,False,True
all_agree,,
False,3,29
True,1,167


In [19]:
preds.eq(cards["intent"], axis=0).groupby(n_answers == 1).mean().T

,False,True
jev-latest,0.84375,0.994048
liquid/d1,0.84375,0.994048
upstage/solar-decide,0.68750,0.994048
jaredpalmer/kev-4b,0.90625,0.994048
perplexity/pplx-decider-v1.2,0.87500,0.994048
cloudflare/clef,0.87500,0.994048
laya,0.18750,0.994048
nimble,0.78125,0.994048


## OpenRouter's Native Decisions Endpoint
- Slides 16-17
- `POST /api/alpha/decisions` takes the same `model`, `state` and `questions`
- Extra OpenRouter fields: `provider` routing, here `zdr` (zero data retention endpoints only)
- The raw response also carries `usage.cost` and the `provider` that served it

In [20]:
body = {
    "model": "jev-latest",
    "state": msg,
    "questions": {"intent": INTENT.model_dump()},
    "provider": {"zdr": True},
}
resp = httpx.post(
    "https://openrouter.ai/api/alpha/decisions",
    headers={"Authorization": "Bearer " + os.environ["TYPESAFE_API_KEY"]},
    json=body,
)
resp.json()

{'model': 'typesafe/jev-1.14-20261014',
 'answers': {'intent': {'type': 'choice',
   'choice': 'card_arrival',
   'probabilities': {'card_arrival': 0.9,
    'activate_my_card': 0,
    'change_pin': 0,
    'card_about_to_expire': 0,
    'lost_or_stolen_card': 0.1,
    'compromised_card': 0,
    'card_not_working': 0,
    'declined_card_payment': 0,
    'card_payment_fee_charged': 0,
    'card_payment_wrong_exchange_rate': 0},
   'confidence': 0.89}},
 'usage': {'input_tokens': 545, 'output_tokens': 121, 'cost': 2.289e-05},
 'id': 'gen-dec-1792250663-zhoxGwgnN9PfS5jvSmhF',
 'provider': 'TypeSafe'}

## Cheat Sheet

| Need | Code |
|---|---|
| Hosted model through OpenRouter | `TypeSafeClient()` with `TYPESAFE_BASE_URL=https://openrouter.ai/api` |
| Pick a provider's model | `client.system_one(state, questions, model="liquid/d1")` |
| Exact version that answered | `r.model` |
| Ollaya (Laya) | `TypeSafeClient(base_url="http://localhost:11435", api_key="local")`, `model="laya"` |
| Ollama (nimble, tev1) | `TypeSafeClient(base_url="http://localhost:11434", api_key="local")`, `model="nimble"` |
| llama.cpp `llama-server` | `TypeSafeClient(base_url="http://localhost:9931", api_key="local")` |
| Liquid direct | `TypeSafeClient(base_url="https://api.liquid.ai", api_key=os.environ["LIQUID_API_KEY"])`, `model="d1"` |
| Perplexity direct (own path) | `POST https://api.perplexity.ai/v1/decisions`, `model="pplx-decider-v1.2"` |
| OpenRouter native endpoint | `POST https://openrouter.ai/api/alpha/decisions` with `provider={...}` |
| Hosted model IDs on OpenRouter | `jev-latest`, `liquid/d1`, `upstage/solar-decide`, `jaredpalmer/kev-4b`, `perplexity/pplx-decider-v1.2`, `cloudflare/clef` |
| Agreement across models | `preds.nunique(axis=1)` |
| Majority answer | `preds.mode(axis=1)[0]` |

## Wrap-Up
- Slides 18-23: OpenAI's own Decisions API; generation versus decision; choosing a backend; portability; base URLs and model IDs